In [7]:
import random
import time
from typing import Callable, Literal, Optional, TypedDict
from langgraph.graph import END, START, StateGraph


# ==========================================
# 1. Day 24 扩展后的 AgentState (包含 probe 标记)
# ==========================================
class AgentState(TypedDict):
    employee_id: str
    amount: float

    # API Attempt 状态
    api_status: Optional[int]
    error_msg: Optional[str]

    # Half-Open 探测身份标记 (关键架构修补)
    is_half_open_probe: bool

    # Retry 策略状态
    retry_count: int
    max_retries: int
    retry_delay: float

    # Deadline 策略状态
    deadline: float
    request_timeout: float

    # External Interruption 状态
    cancelled: bool

    # Policy 决策输出
    policy_action: Optional[
        Literal[
            "ALLOW",
            "FAST_FAIL",
            "SUCCESS",
            "RETRY",
            "FALLBACK",
            "DEADLINE_EXCEEDED",
            "CANCELLED",
            "FATAL_ERROR",
        ]
    ]

    # 执行结果
    result: Optional[str]


# ==========================================
# 2. 依赖注入 (Injectable Side-effects)
# ==========================================
clock_fn: Callable[[], float] = time.time
sleep_fn: Callable[[float], None] = time.sleep
random_fn: Callable[[float, float], float] = random.uniform


# ==========================================
# 3. 共享 Circuit Breaker 实例
# ==========================================
class CircuitBreaker:

    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"  # "CLOSED", "OPEN", "HALF_OPEN"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = clock_fn()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = clock_fn()


# 全局共享 Breaker 实例
hr_api_breaker = CircuitBreaker()


# ==========================================
# 4. 5 大核心 Node 实现
# ==========================================
def circuit_breaker_gate_node(state: AgentState) -> dict:
    """1. 入口门控：捕获允许状态及 HALF_OPEN 身份"""
    allowed = hr_api_breaker.can_call()
    if not allowed:
        return {"policy_action": "FAST_FAIL", "is_half_open_probe": False}

    # 在 can_call() 触发后精准捕获当前是否为 HALF_OPEN 探测流量
    is_probe = hr_api_breaker.state == "HALF_OPEN"
    return {"policy_action": "ALLOW", "is_half_open_probe": is_probe}


def call_hr_api_node(state: AgentState) -> dict:
    """2. 单次 API 尝试"""
    status = state.get("api_status", 503)
    error_msg = "OK" if status == 200 else f"HTTP Error {status}"
    return {"api_status": status, "error_msg": error_msg}


def policy_node(state: AgentState) -> dict:
    """3. 组合策略决策中心"""
    # 1. 外部取消优先判定
    if state.get("cancelled", False):
        return {"policy_action": "CANCELLED"}

    status = state.get("api_status")

    # 2. 成功路径
    if status == 200:
        hr_api_breaker.record_success()
        return {"policy_action": "SUCCESS", "result": "API Call Succeeded"}

    # 失败反馈给共享 Breaker
    hr_api_breaker.record_failure()

    # 3. 不可重试致命错误
    if status in (400, 401, 403, 404, 422, 501):
        return {"policy_action": "FATAL_ERROR", "result": f"Fatal Error: {status}"}

    # 核心修补：仅当“当前请求本身是 HALF_OPEN Probe”且失败时才进入 FALLBACK
    # 如果是普通的请求把 CLOSED 打成了 OPEN，它依然可以凭自己的 Retry Policy 重试
    if state.get("is_half_open_probe", False):
        return {
            "policy_action": "FALLBACK",
            "result": "Half-Open Probe Failed -> Fallback",
        }

    # 4. 重试次数预算检查
    if state["retry_count"] >= state["max_retries"]:
        return {
            "policy_action": "FALLBACK",
            "result": "Max Retries Reached -> Fallback",
        }

    # 5. 计算 Backoff + Jitter
    backoff = state.get("retry_delay", 1.0) * (2 ** state["retry_count"])
    jitter = random_fn(0.0, 0.5)
    computed_delay = backoff + jitter

    # 6. Deadline 预算检查
    now = clock_fn()
    remaining_budget = state["deadline"] - now
    required_budget = computed_delay + state["request_timeout"] + 0.1

    if remaining_budget < required_budget:
        return {
            "policy_action": "DEADLINE_EXCEEDED",
            "result": "Deadline Exceeded -> Fallback",
        }

    return {"policy_action": "RETRY", "retry_delay": computed_delay}


def retry_wait_node(state: AgentState) -> dict:
    """4. 执行退避等待"""
    sleep_fn(state["retry_delay"])
    return {"retry_count": state["retry_count"] + 1}


def fallback_node(state: AgentState) -> dict:
    """5. 降级节点"""
    return {
        "result": f"Fallback Executed. Trigger Action: {state.get('policy_action')}"
    }


# ==========================================
# 5. Router 与 Graph 编排
# ==========================================
def route_gate(state: AgentState) -> str:
    if state["policy_action"] == "ALLOW":
        return "call_hr_api"
    return "fallback"


def route_policy(state: AgentState) -> str:
    action = state["policy_action"]
    if action == "SUCCESS":
        return "end"
    if action == "RETRY":
        return "retry_wait"
    if action in ("FALLBACK", "DEADLINE_EXCEEDED"):
        return "fallback"
    if action in ("CANCELLED", "FATAL_ERROR"):
        return "end"
    return "end"


builder = StateGraph(AgentState)

builder.add_node("circuit_breaker_gate", circuit_breaker_gate_node)
builder.add_node("call_hr_api", call_hr_api_node)
builder.add_node("policy", policy_node)
builder.add_node("retry_wait", retry_wait_node)
builder.add_node("fallback", fallback_node)

builder.set_entry_point("circuit_breaker_gate")

builder.add_conditional_edges(
    "circuit_breaker_gate",
    route_gate,
    {"call_hr_api": "call_hr_api", "fallback": "fallback"},
)

builder.add_edge("call_hr_api", "policy")

builder.add_conditional_edges(
    "policy",
    route_policy,
    {
        "end": END,
        "retry_wait": "retry_wait",
        "fallback": "fallback",
    },
)

# 关键回环：retry_wait 直连 call_hr_api，不通过 Gate
builder.add_edge("retry_wait", "call_hr_api")
builder.add_edge("fallback", END)

graph = builder.compile()


# ==========================================
# 6. 场景验证
# ==========================================
if __name__ == "__main__":
    # Mock sleep_fn 避免测试真实等待
    sleep_fn = lambda x: None

    print("--- Scenario 1: Retry Loop until Max Retries Exceeded ---")
    hr_api_breaker = CircuitBreaker()
    state_1: AgentState = {
        "employee_id": "EMP_001",
        "amount": 100.0,
        "api_status": 503,
        "error_msg": None,
        "is_half_open_probe": False,
        "retry_count": 0,
        "max_retries": 2,
        "retry_delay": 0.1,
        "deadline": clock_fn() + 10.0,
        "request_timeout": 1.0,
        "cancelled": False,
        "policy_action": None,
        "result": None,
    }
    res_1 = graph.invoke(state_1)
    print(
        f"Action: {res_1.get('policy_action')} | Retries:"
        f" {res_1.get('retry_count')} | Result: {res_1.get('result')}\n"
    )

    print("--- Scenario 2: Deadline Exceeded ---")
    hr_api_breaker = CircuitBreaker()
    state_2: AgentState = {
        "employee_id": "EMP_002",
        "amount": 200.0,
        "api_status": 500,
        "error_msg": None,
        "is_half_open_probe": False,
        "retry_count": 1,
        "max_retries": 5,
        "retry_delay": 2.0,
        "deadline": clock_fn() + 1.5,
        "request_timeout": 1.0,
        "cancelled": False,
        "policy_action": None,
        "result": None,
    }
    res_2 = graph.invoke(state_2)
    print(
        f"Action: {res_2.get('policy_action')} | Result:"
        f" {res_2.get('result')}\n"
    )

    print("--- Scenario 3: External Cancellation ---")
    state_3: AgentState = {
        "employee_id": "EMP_003",
        "amount": 300.0,
        "api_status": 200,
        "error_msg": None,
        "is_half_open_probe": False,
        "retry_count": 0,
        "max_retries": 3,
        "retry_delay": 0.1,
        "deadline": clock_fn() + 10.0,
        "request_timeout": 1.0,
        "cancelled": True,
        "policy_action": None,
        "result": None,
    }
    res_3 = graph.invoke(state_3)
    print(
        f"Action: {res_3.get('policy_action')} | Result:"
        f" {res_3.get('result')}\n"
    )

    print(
        "--- Scenario 4 (关键新增): Breaker Threshold=1 时，旧 Workflow 允许"
        " Retry，新 Workflow 被 Fast Fail ---"
    )
    hr_api_breaker = CircuitBreaker(failure_threshold=1, cooldown=60.0)

    # Workflow A 初始进入 (Breaker 为 CLOSED)
    state_wf_a: AgentState = {
        "employee_id": "WF_A",
        "amount": 100.0,
        "api_status": 503,  # Attempt 1 会失败
        "error_msg": None,
        "is_half_open_probe": False,
        "retry_count": 0,
        "max_retries": 1,
        "retry_delay": 0.1,
        "deadline": clock_fn() + 10.0,
        "request_timeout": 1.0,
        "cancelled": False,
        "policy_action": None,
        "result": None,
    }

    # 执行模拟：WF_A 尝试 Attempt 1 失败后触发 Breaker 变为 OPEN，但 policy 仍给它 RETRY 判定
    res_wf_a_step1 = circuit_breaker_gate_node(state_wf_a)
    state_wf_a.update(res_wf_a_step1)
    state_wf_a.update(call_hr_api_node(state_wf_a))
    res_policy_wf_a = policy_node(state_wf_a)

    print(
        f"[WF_A] Attempt 1 失败后 Breaker 状态: {hr_api_breaker.state} (已变成"
        " OPEN)"
    )
    print(
        f"[WF_A] Policy Action: {res_policy_wf_a.get('policy_action')} (依然成功拿到"
        " RETRY 权限，非 FALLBACK)\n"
    )

    # 此时新的 Workflow B 尝试进入 Gate
    state_wf_b: AgentState = {
        "employee_id": "WF_B",
        "amount": 50.0,
        "api_status": 200,
        "error_msg": None,
        "is_half_open_probe": False,
        "retry_count": 0,
        "max_retries": 3,
        "retry_delay": 0.1,
        "deadline": clock_fn() + 10.0,
        "request_timeout": 1.0,
        "cancelled": False,
        "policy_action": None,
        "result": None,
    }
    res_wf_b = graph.invoke(state_wf_b)
    print(
        f"[WF_B] 尝试进入 Gate 的 Policy Action:"
        f" {res_wf_b.get('policy_action')} (由于 Breaker 为 OPEN，直接"
        " FAST_FAIL)"
    )

--- Scenario 1: Retry Loop until Max Retries Exceeded ---
Action: FALLBACK | Retries: 2 | Result: Fallback Executed. Trigger Action: FALLBACK

--- Scenario 2: Deadline Exceeded ---
Action: DEADLINE_EXCEEDED | Result: Fallback Executed. Trigger Action: DEADLINE_EXCEEDED

--- Scenario 3: External Cancellation ---
Action: CANCELLED | Result: None

--- Scenario 4 (关键新增): Breaker Threshold=1 时，旧 Workflow 允许 Retry，新 Workflow 被 Fast Fail ---
[WF_A] Attempt 1 失败后 Breaker 状态: OPEN (已变成 OPEN)
[WF_A] Policy Action: RETRY (依然成功拿到 RETRY 权限，非 FALLBACK)

[WF_B] 尝试进入 Gate 的 Policy Action: FAST_FAIL (由于 Breaker 为 OPEN，直接 FAST_FAIL)
